# Grab App Reviews Sentiment Analysis

## SVM + TF-IDF

### Import Libraries

In [1]:
!pip install -q sastrawi
import pandas as pd
import numpy as np
import csv
import requests
import json
from io import StringIO
import matplotlib.pyplot as plt
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from wordcloud import WordCloud
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load Dataset

In [2]:
url = 'https://raw.githubusercontent.com/bluga404/sa-grab/main/reviews_traveloka.csv'
reviews = pd.read_csv(url)

# reviews and columns
total_reviews, total_columns  = reviews.shape
print(f"Total reviews: {total_reviews}, Total columns: {total_columns}")
print("-"*50)
reviews.info()

Total reviews: 64839, Total columns: 11
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64839 entries, 0 to 64838
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              64839 non-null  object
 1   userName              64839 non-null  object
 2   userImage             64839 non-null  object
 3   content               64839 non-null  object
 4   score                 64839 non-null  int64 
 5   thumbsUpCount         64839 non-null  int64 
 6   reviewCreatedVersion  64839 non-null  object
 7   at                    64839 non-null  object
 8   replyContent          64826 non-null  object
 9   repliedAt             64839 non-null  object
 10  appVersion            64839 non-null  object
dtypes: int64(2), object(9)
memory usage: 5.4+ MB


### Preprocessing

In [3]:
# stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stem_cache = {}

# stopwords
stopword = set(stopwords.words('indonesian')) \
            | set(stopwords.words('english'))

# regex precompiled
mention = re.compile(r'@[A-Za-z0-9]+')
hashtag = re.compile(r'#[A-Za-z0-9]+')
rt = re.compile(r'\bRT\b')
url = re.compile(r'http\S+')
number = re.compile(r'\d+')
symbol = re.compile(r'[^\w\s]')

slang_url = 'https://raw.githubusercontent.com/okkyibrohim/id-abusive-language-detection/master/kamusalay.csv'
slang_df = pd.read_csv(slang_url, header=None, names=['slang', 'formal'])

slangwords = dict(zip(
    slang_df['slang'].str.lower(),
    slang_df['formal']
))

In [4]:
def cleaningText(text):
    text = mention.sub('', text)
    text = hashtag.sub('', text)
    text = rt.sub('', text)
    text = url.sub('', text)
    text = number.sub('', text)
    text = symbol.sub('', text)
    return text.strip().lower()

def stem_word(word):
    if word in stem_cache:
        return stem_cache[word]

    stemmed = stemmer.stem(word)
    stem_cache[word] = stemmed
    return stemmed

def fixSlang(text):
    # split text to words
    words = text.split()
    # slang + stopword in one pass
    words = [
        stem_word(slangwords.get(w, w))
        for w in words
        if w not in stopword
    ]
    return ' '.join(words)

In [5]:
reviews['final_text'] = reviews['content'].apply(cleaningText)
reviews['final_text'] = reviews['final_text'].apply(fixSlang)
reviews.to_csv('clean_reviews_traveloka.csv', index=False)

In [6]:
reviews = pd.read_csv('clean_reviews_traveloka.csv')
reviews = reviews.dropna()
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 62190 entries, 0 to 64838
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              62190 non-null  object
 1   userName              62190 non-null  object
 2   userImage             62190 non-null  object
 3   content               62190 non-null  object
 4   score                 62190 non-null  int64 
 5   thumbsUpCount         62190 non-null  int64 
 6   reviewCreatedVersion  62190 non-null  object
 7   at                    62190 non-null  object
 8   replyContent          62190 non-null  object
 9   repliedAt             62190 non-null  object
 10  appVersion            62190 non-null  object
 11  final_text            62190 non-null  object
dtypes: int64(2), object(10)
memory usage: 6.2+ MB


### Labeling

In [7]:
def load_lexicons():
    # Positive lexicon
    positive_lexicon = {}
    pos_url = 'https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv'
    response = requests.get(pos_url)

    if response.status_code == 200:
        reader = csv.reader(StringIO(response.text), delimiter=',')
        for row in reader:
            positive_lexicon[row[0]] = int(row[1])
    else:
        raise Exception("Failed to fetch positive lexicon data")

    # Negative lexicon
    negative_lexicon = {}
    neg_url = 'https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv'
    response = requests.get(neg_url)

    if response.status_code == 200:
        reader = csv.reader(StringIO(response.text), delimiter=',')
        for row in reader:
            negative_lexicon[row[0]] = int(row[1])
    else:
        raise Exception("Failed to fetch negative lexicon data")

    return positive_lexicon, negative_lexicon

def sentiment_label(text, pos_lex, neg_lex):
    score = 0
    words = text.split()

    for w in words:
        score += pos_lex.get(w, 0)
        score += neg_lex.get(w, 0)

    return 1 if score > 0 else -1 if score < 0 else 0

In [8]:
pos_lex, neg_lex = load_lexicons()
reviews['label'] = reviews['final_text'].apply(
    lambda x: sentiment_label(x, pos_lex, neg_lex)
)
print(reviews['label'].value_counts())

label
 1    30028
-1    23132
 0     9030
Name: count, dtype: int64


In [9]:
reviews.to_csv('labeled_reviews_traveloka.csv', index=False)

### Data Split and Feature Extraction

In [10]:
X = reviews['final_text']
y = reviews['label']

tfidf = TfidfVectorizer(max_features=5000, min_df=17, max_df=0.8 )
X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

### Modeling

In [11]:
svm = LinearSVC(C=1.0, class_weight='balanced', random_state=42)
svm.fit(X_train.toarray(), y_train)

LinearSVC(class_weight='balanced', random_state=42)

In [12]:
y_pred_train_svm = svm.predict(X_train.toarray())
y_pred_test_svm = svm.predict(X_test.toarray())

In [13]:
accuracy_train_svm = accuracy_score(y_pred_train_svm, y_train)
accuracy_test_svm = accuracy_score(y_pred_test_svm, y_test)

print('SVM + TF-IDF - Train accuracy:', accuracy_train_svm)
print('SVM + TF-IDF - Test accuracy:', accuracy_test_svm)

SVM + TF-IDF - Train accuracy: 0.9630625043070774
SVM + TF-IDF - Test accuracy: 0.9489735756016509


### Testing Model / Inference

In [14]:
texts = [
    "Driver grab sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

X_new = tfidf.transform(texts)
y_pred_new = svm.predict(X_new.toarray())

label_map = {
    -1: "Negative",
     0: "Netral",
     1: "Positive"
}

for text, label in zip(texts, y_pred_new):
    print("Text:", text)
    print("Prediction:", label_map[label])
    print("-" * 50)

Text: Driver grab sangat buruk dan mengecewakan
Prediction: Negative
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive
--------------------------------------------------
Text: saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Negative
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive
--------------------------------------------------


## BiLSTM + Word2Vec

### Import Libraries

In [15]:
!pip install -q gensim
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
2026-01-26 08:48:42.356123: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769417322.537274      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769417322.589429      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769417323.002388      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid l

### Load Dataset

In [16]:
reviews = pd.read_csv('clean_reviews_traveloka.csv')
reviews = reviews.dropna()
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 62190 entries, 0 to 64838
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              62190 non-null  object
 1   userName              62190 non-null  object
 2   userImage             62190 non-null  object
 3   content               62190 non-null  object
 4   score                 62190 non-null  int64 
 5   thumbsUpCount         62190 non-null  int64 
 6   reviewCreatedVersion  62190 non-null  object
 7   at                    62190 non-null  object
 8   replyContent          62190 non-null  object
 9   repliedAt             62190 non-null  object
 10  appVersion            62190 non-null  object
 11  final_text            62190 non-null  object
dtypes: int64(2), object(10)
memory usage: 6.2+ MB


### Data Split and Feature Extraction

In [17]:
# get text
texts = reviews['final_text']
# convert text to list
texts = texts.tolist()
# apply tokenization for every sentence in text list
tokenized_text = [word_tokenize(sentence) for sentence in texts]

In [18]:
w2v_model = Word2Vec(
    window = 10,
    min_count = 5,
    workers = 4,
    epochs = 10
)

w2v_model.build_vocab(tokenized_text, progress_per=1000)
w2v_model.train(tokenized_text, total_examples=w2v_model.corpus_count, epochs=w2v_model.epochs)
w2v_model.save('word2vec-indo.model')

w2v_model = Word2Vec.load('word2vec-indo.model')
w2v_model.wv.most_similar("mantap")

[('mantab', 0.9406041502952576),
 ('sip', 0.8843321204185486),
 ('keren', 0.8836347460746765),
 ('poko', 0.8528846502304077),
 ('mantul', 0.8523291349411011),
 ('mantappp', 0.8380498886108398),
 ('mantapp', 0.8378373384475708),
 ('jos', 0.8353772163391113),
 ('oke', 0.8288429379463196),
 ('holiday', 0.8260272145271301)]

In [19]:
print("vocab length:", len(w2v_model.wv.key_to_index))

length = [len(tokens) for tokens in tokenized_text]
print(f"longest: {max(length)} words")
print(f"shortest: {min(length)} words")
print(f"median: {np.median(length)} words")

vocab length: 3503
longest: 186 words
shortest: 0 words
median: 2.0 words


In [20]:
vocab_size = 10000
max_len = 50
embedding_dim = w2v_model.vector_size

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<oov>")
tokenizer.fit_on_texts(tokenized_text)
word_index = tokenizer.word_index
print(f"there are {len(word_index)} unique tokens")

there are 24167 unique tokens


In [21]:
sequences = tokenizer.texts_to_sequences(tokenized_text)
X_data = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# +1 cause index 0 reserved for padding
num_words = min(vocab_size, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))

found_words = 0
for word, i in word_index.items():
    if i > vocab_size:
        continue

    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
        found_words += 1
    else:
        pass

print(f"mapped {found_words} from word2vec to keras")

mapped 3503 from word2vec to keras


In [22]:
pos_lex, neg_lex = load_lexicons()
reviews['label'] = reviews['final_text'].apply(
    lambda x: sentiment_label(x, pos_lex, neg_lex)
)
print(reviews['label'].value_counts())

label
 1    30028
-1    23132
 0     9030
Name: count, dtype: int64


In [23]:
y = reviews['label'].values
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y,
    test_size = 0.3,
    stratify=y
)

# shift labels to be 0, 1, 2
y_train = y_train + 1
y_test = y_test + 1

In [24]:
print(f"training data shape: {X_train.shape}")
print(f"test data shape: {X_test.shape}")

training data shape: (43533, 50)
test data shape: (18657, 50)


### Modeling

In [25]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=num_words,
        output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        input_length=max_len,
        trainable=True
    )
)

# Bi-LSTM Layer
model.add(Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=l2(0.001))))
model.add(Dropout(0.5))

model.add(Bidirectional(LSTM(32, return_sequences=False, kernel_regularizer=l2(0.001))))
model.add(Dropout(0.5))

# Dense Layer
model.add(Dense(32, activation='relu'))

# Output Layer
model.add(Dense(3, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1769417340.947333      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [26]:
optimizer = Adam(learning_rate=1e-4)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [27]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50


I0000 00:00:1769417347.046844     131 cuda_dnn.cc:529] Loaded cuDNN version 91002


1225/1225 ━━━━━━━━━━━━━━━━━━━━ 28s 17ms/step - accuracy: 0.6771 - loss: 1.1871 - val_accuracy: 0.8886 - val_loss: 0.5691
Epoch 2/50
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.8966 - loss: 0.5282 - val_accuracy: 0.9095 - val_loss: 0.4404
Epoch 3/50
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9240 - loss: 0.3833 - val_accuracy: 0.9295 - val_loss: 0.3596
Epoch 4/50
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9429 - loss: 0.2985 - val_accuracy: 0.9389 - val_loss: 0.3003
Epoch 5/50
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9508 - loss: 0.2550 - val_accuracy: 0.9460 - val_loss: 0.2685
Epoch 6/50
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9598 - loss: 0.2134 - val_accuracy: 0.9476 - val_loss: 0.2476
Epoch 7/50
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9655 - loss: 0.1889 - val_accuracy: 0.9479 - val_loss: 0.2550
Epoch 8/50
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9704 - loss: 0.16

In [28]:
results = model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {results[0]:.4f}")
print(f"Test Accuracy: {results[1]*100:.2f}%")

584/584 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9571 - loss: 0.2105
Test Loss: 0.2092
Test Accuracy: 95.51%


### Testing Model / Inference

In [29]:
texts = [
    "Driver grab sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

In [30]:
processed_texts = []

for text in texts:
    text = str(text).lower()
    fixed = fixSlang(text)
    processed_texts.append(fixed)

final_tokens = [word_tokenize(t) for t in processed_texts]
seqs = tokenizer.texts_to_sequences(final_tokens)
X_new = pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')

predictions = model.predict(X_new)
y_pred_indices = np.argmax(predictions, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step


In [31]:
label_map = {
    -1: "Negative",
     0: "Netral",
     1: "Positive"
}

for text, index in zip(texts, y_pred_indices):
    # Convert 0,1,2 back to -1,0,1
    original_label = index - 1
    sentiment = label_map[original_label]

    print(f"Text: {text}")
    print(f"Prediction: {sentiment}")
    print("-" * 50)

Text: Driver grab sangat buruk dan mengecewakan
Prediction: Negative
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive
--------------------------------------------------
Text: saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Negative
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive
--------------------------------------------------


## IndoBERT-Base

### Import Libraries

In [32]:
!pip install -U -q transformers
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import tf_keras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.3 MB/s eta 0:00:00


### Load Dataset

In [33]:
reviews = pd.read_csv('clean_reviews_traveloka.csv')
reviews = reviews.dropna()
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 62190 entries, 0 to 64838
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              62190 non-null  object
 1   userName              62190 non-null  object
 2   userImage             62190 non-null  object
 3   content               62190 non-null  object
 4   score                 62190 non-null  int64 
 5   thumbsUpCount         62190 non-null  int64 
 6   reviewCreatedVersion  62190 non-null  object
 7   at                    62190 non-null  object
 8   replyContent          62190 non-null  object
 9   repliedAt             62190 non-null  object
 10  appVersion            62190 non-null  object
 11  final_text            62190 non-null  object
dtypes: int64(2), object(10)
memory usage: 6.2+ MB


In [34]:
pos_lex, neg_lex = load_lexicons()
reviews['label'] = reviews['final_text'].apply(
    lambda x: sentiment_label(x, pos_lex, neg_lex)
)
print(reviews['label'].value_counts())

label
 1    30028
-1    23132
 0     9030
Name: count, dtype: int64


### Data Split and Feature Extraction

In [35]:
X = reviews['final_text'].astype(str).tolist()
y = reviews['label'].values + 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
      stratify=y
)

In [36]:
tokenizer = AutoTokenizer.from_pretrained("sarahlintang/IndoBERT")

def fast_encode(texts, tokenizer, max_len=60):
    inputs = tokenizer(
        texts,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )
    return dict(inputs)

train_data = fast_encode(X_train, tokenizer)
test_data = fast_encode(X_test, tokenizer)

config.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


### Modeling

In [37]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "sarahlintang/IndoBERT",
    num_labels=3,
    from_pt=True
)

optimizer = tf_keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
model.summary()

pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  112921344 
                                                                 
 dropout_37 (Dropout)        multiple                  0         
                                                                 
 classifier (Dense)          multiple                  2307      
                                                                 
Total params: 112923651 (430.77 MB)
Trainable params: 112923651 (430.77 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [38]:
history = model.fit(
    train_data,
    y_train,
    validation_split=0.1,
    epochs=4,
    batch_size=16
)

Epoch 1/4


I0000 00:00:1769417692.827394     130 service.cc:152] XLA service 0x7b920c051f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769417692.827470     130 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1769417693.087650     130 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2449/2449 [==============================] - 425s 153ms/step - loss: 0.2490 - accuracy: 0.9104 - val_loss: 0.1631 - val_accuracy: 0.9437
Epoch 2/4
2449/2449 [==============================] - 357s 146ms/step - loss: 0.1412 - accuracy: 0.9503 - val_loss: 0.1348 - val_accuracy: 0.9511
Epoch 3/4
2449/2449 [==============================] - 357s 146ms/step - loss: 0.1017 - accuracy: 0.9640 - val_loss: 0.1352 - val_accuracy: 0.9559
Epoch 4/4
2449/2449 [==============================] - 357s 146ms/step - loss: 0.0793 - accuracy: 0.9713 - val_loss: 0.1295 - val_accuracy: 0.9603


In [39]:
test_accuracy, test_loss = model.evaluate(test_data, y_test)

584/584 [==============================] - 51s 87ms/step - loss: 0.1568 - accuracy: 0.9544


### Testing Model / Inference

In [40]:
texts = [
    "Driver gofood sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

In [41]:
processed_texts = []

for text in texts:
    text = str(text).lower()
    fixed = fixSlang(text)
    processed_texts.append(fixed)

inputs = tokenizer(
    processed_texts,
    max_length=60,
    padding='max_length',
    truncation=True,
    return_tensors='tf'
)

logits = model.predict(dict(inputs)).logits

probabilities = tf.nn.softmax(logits).numpy()
y_pred_indices = np.argmax(probabilities, axis=1)

1/1 [==============================] - 3s 3s/step


In [42]:
label_map = {
    -1: "Negative",
     0: "Neutral",
     1: "Positive"
}

for text, index, probs in zip(texts, y_pred_indices, probabilities):
    original_label = index - 1
    sentiment = label_map[original_label]
    confidence = probs[index] * 100

    print(f"Text: {text}")
    print(f"Prediction: {sentiment} ({confidence:.2f}%)")
    print("-" * 50)

Text: Driver gofood sangat buruk dan mengecewakan
Prediction: Negative (99.97%)
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative (99.98%)
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive (99.96%)
--------------------------------------------------
Text: saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Negative (90.05%)
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive (99.96%)
--------------------------------------------------


In [43]:
!pip freeze > requirements.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
